# **Project: Cambodian Tourism Chatbot (RNN)**
---


### **1. Project Overview**

> In this project, you will build a Natural Language Processing (NLP) model using a **Simple Recurrent Neural Network (RNN)**. The goal is to create a chatbot that can answer common questions about tourism in Cambodia, covering destinations, food, and travel logistics
---


### **2. Environment Setup**

Import all necessary libraries for data manipulation, visualization, and deep learning.


In [2]:
# Import pandas, numpy, matplotlib, and tensorflow
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

### **3. Data Acquisition & Exploration**
#### **3.1 Load Dataset**
**Instructions**: Load the `cambodia_tourism_dataset_large.csv` file and display the first 10 rows.


In [3]:
# TODO: Load the CSV file
df = pd.read_csv("cambodia_tourism_dataset_large.csv")

# TODO: Display the head of the dataframe
df.head(10)

,question,answer
0,can you explain where is angkor wat specifically.,angkor wat is in siem reap cambodia
1,give me details on what is angkor wat for me.,it is a famous temple in cambodia
2,hey what to do in siem reap specifically.,you can visit temples and night market
3,do you know where is bayon temple immediately.,bayon temple is in the center of angkor thom
4,do you know what is ta prohm specifically.,it is the jungle temple with giant trees in si...
5,hi how much is angkor pass thank you.,a 1-day pass is 37 dollars and a 3-day pass is...
6,do you know where to watch sunset in siem reap...,phnom bakheng is the most popular spot for sunset
7,what is pub street?,it is the nightlife hub of siem reap with bars...
8,is tonle sap near siem reap specifically.,yes it is the largest freshwater lake near sie...
9,can you tell me best time to visit temples,early morning at sunrise is the best time


#### **3.2 Dataset Analysis**
**Instructions**: Perform basic EDA. Calculate the total number of samples and the maximum length of the questions and answers.


In [4]:
# TODO: Print total number of rows
print(f"Total samples: {len(df)}")

# TODO: Find the maximum length (word count) of the 'question' and 'answer' columns
df['question_length'] = df['question'].apply(lambda x: len(str(x).split()))
df['answer_length'] = df['answer'].apply(lambda x: len(str(x).split()))

print(f"Max question length (words): {df['question_length'].max()}")
print(f"Max answer length (words): {df['answer_length'].max()}")
print(f"Average question length: {df['question_length'].mean():.2f}")
print(f"Average answer length: {df['answer_length'].mean():.2f}")

Total samples: 2000
Max question length (words): 14
Max answer length (words): 13
Average question length: 8.68
Average answer length: 8.70


### **4. Data Preprocessing**
#### **4.1 Tokenization**
Convert text into numerical format. Fit a tokenizer on both questions and answers to build a shared vocabulary.


In [5]:
# Correct column names from generated CSV
inputs = df["question"].astype(str).tolist()
responses = df["answer"].astype(str).tolist()

In [6]:
# Tokenization
## Initialize the Tokenizer
tokenizer = Tokenizer(filters='') # Keep it simple for RNN
tokenizer.fit_on_texts(inputs + responses) # Fit on text
vocab_size = len(tokenizer.word_index) + 1

X_seq = tokenizer.texts_to_sequences(inputs)
y_seq = tokenizer.texts_to_sequences(responses)

In [7]:
# 1. See how many unique words were found
print(f"Total words in vocabulary: {vocab_size}")

# 2. Look at the first 10 words in the dictionary
print("Word Index Sample:", dict(list(tokenizer.word_index.items())[:10]))

# 3. Verify the conversion
sample_text = inputs[0]
sample_seq = X_seq[0]
print(f"\nOriginal: {sample_text}")
print(f"Numerical: {sample_seq}")

Total words in vocabulary: 421
Word Index Sample: {'is': 1, 'the': 2, 'in': 3, 'it': 4, 'you': 5, 'what': 6, 'to': 7, 'a': 8, 'for': 9, 'where': 10}

Original: can you explain where is angkor wat specifically.
Numerical: [13, 5, 24, 10, 1, 19, 37, 38]


#### Explanation
Since neural networks like a `SimpleRNN` cannot "read" text, we must convert words into unique integers.

**1. `Tokenizer(filters='')`**
By default, a tokenizer removes punctuation (like `?`, `!`, and `.`). By setting `filters=''`, we tell the model to keep the text exactly as it is. This is important if you want the chatbot to recognize the difference between a statement and a question.

**2. `tokenizer.fit_on_texts(inputs + responses)`**
This is the "learning" phase for the tokenizer. It scans every word in your dataset and builds a dictionary (a **Word Index**). Every unique word is assigned a specific number.
*   *Example:* "Angkor" becomes `1`, "Wat" becomes `2`, "Siem" becomes `3`, and so on.

**3. `vocab_size = len(tokenizer.word_index) + 1`**
This calculates the total number of unique words the model knows. We add `+1` because the computer reserves the number **`0`** for a special task called "Padding" (filling empty space in short sentences).

**4. `texts_to_sequences`**
This is the final transformation. It takes your actual sentences and replaces the words with the numbers from the dictionary.

---

**Before (Text):**
> `"where is angkor wat"`

**The Word Index (Dictionary):**
> `{'where': 1, 'is': 2, 'angkor': 3, 'wat': 4}`

**After (`X_seq`):**
> `[1, 2, 3, 4]`

---


#### **4.2 Sequencing and Padding**
**Instructions**: Convert the text strings into sequences of integers and pad them so they all have the same length. Explain why padding is necessary for RNNs.


In [8]:
# Padding
# Calculate max_len based on 95th percentile or max to keep sequences uniform
max_len = max(max(len(seq) for seq in X_seq), max(len(seq) for seq in y_seq))

# TODO: Pad the sequences to ensure uniform length for model input
X = pad_sequences(X_seq, maxlen=max_len, padding='post')
y = pad_sequences(y_seq, maxlen=max_len, padding='post')

print(f"X shape: {X.shape}")  # (num_samples, max_len)
print(f"y shape: {y.shape}")  # (num_samples, max_len)
print(f"max_len: {max_len}")


X shape: (2000, 14)
y shape: (2000, 14)
max_len: 14


#### **4.3 Train-Test Split**
**Instructions**: Split the data into 80% training and 20% testing sets to evaluate your model's performance on unseen data.


In [9]:
# TODO: Split X and y using train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 80% train, 20% test
    random_state=42     # For reproducibility
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

Training samples: 1600
Testing samples: 400


In [10]:
# 5. One-hot encoding for the targets
# Note: With large vocab_size, this uses a lot of RAM.
y_train_oh = tf.keras.utils.to_categorical(y_train, num_classes=vocab_size)
y_test_oh = tf.keras.utils.to_categorical(y_test, num_classes=vocab_size)

This step, known as **One-Hot Encoding**, is where we transform our target word IDs into a format that the model can use to calculate "how wrong" its guess was during training.

**1. What is One-Hot Encoding?**
In the previous step, we turned words into integers (e.g., "Angkor" = 3). However, we cannot simply use the number `3` as a final answer for the model to predict. If the model guesses `2` instead of `3`, the math would suggest it was "close." In language, word #2 and word #3 are completely different concepts, not "close" numbers.

**One-Hot Encoding** fixes this by turning each integer into a **binary vector**. Every position in the vector is `0`, except for the specific index of the word, which is `1`.

**Example:**
If our vocabulary is only 5 words: `[apple, banana, angkor, wat, siem]`
*   The word **"angkor"** (index 2) becomes: `[0, 0, 1, 0, 0]`
*   The word **"apple"** (index 0) becomes: `[1, 0, 0, 0, 0]`

**2. Why use `tf.keras.utils.to_categorical`?**
This function automates that transformation. It takes your list of integers (`y_train`) and creates a massive table (matrix) of `0`s and `1`s. This allows the final layer of your RNN (the **Softmax** layer) to output a probability for every single word in your dictionary.

In [11]:
# Look at the first word of the first response as an integer
print("Integer format:", y_train[0][0])

# Look at that same word after One-Hot Encoding
print("One-Hot format (first 10 slots):", y_train_oh[0][0][:10])

# Verify the shape of the data
print("Shape of One-Hot data:", y_train_oh.shape)

Integer format: 4
One-Hot format (first 10 slots): [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
Shape of One-Hot data: (1600, 14, 421)


### **5. Model Architecture Design**
#### **5.1 Design Strategy**
**Instructions**: Outline and define your model architecture. You are required to use an **Embedding Layer** and a **SimpleRNN**.

In [15]:
# TODO: Define the Sequential model
model = Sequential()

# TODO: Add Embedding layer (with mask_zero=True)
model.add(Embedding(vocab_size, 64, input_length=max_len, mask_zero=True))

# TODO: Add SimpleRNN layer
model.add(SimpleRNN(128, return_sequences=True))

# TODO: Add Dense layer with Softmax activation
model.add(Dense(vocab_size, activation='softmax'))

# View model summary
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

#### **5.2 Compilation**
**Instructions**: Compile the model

In [16]:
# TODO: Compile the model
model.compile(
    loss='categorical_crossentropy',  # For multi-class classification
    optimizer='adam',                  # Adaptive learning rate optimizer
    metrics=['accuracy']               # Track accuracy during training
)

### **6. Training and Evaluation**
#### **6.1 Model Training**
**Instructions**: Train the model for at least 50-100 epochs. Use the test set as validation data.


In [17]:
# TODO: Run model.fit() and save the history
history = model.fit(
    X_train,
    y_train_oh,                    # One-hot encoded targets
    epochs=100,                    # Number of training iterations
    batch_size=32,                 # Samples per gradient update
    validation_data=(X_test, y_test_oh),  # Evaluate on test set each epoch
    verbose=1                      # Show progress
)

Epoch 1/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 6s 61ms/step - accuracy: 0.1446 - loss: 5.1697 - val_accuracy: 0.1982 - val_loss: 4.5001
Epoch 2/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 60ms/step - accuracy: 0.2284 - loss: 4.1518 - val_accuracy: 0.2448 - val_loss: 3.9029
Epoch 3/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.2447 - loss: 3.7131 - val_accuracy: 0.2487 - val_loss: 3.6332
Epoch 4/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.2483 - loss: 3.4937 - val_accuracy: 0.2530 - val_loss: 3.4516
Epoch 5/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.2559 - loss: 3.3242 - val_accuracy: 0.2541 - val_loss: 3.3053
Epoch 6/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.2600 - loss: 3.2110 - val_accuracy: 0.2564 - val_loss: 3.2134
Epoch 7/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.2623 - loss: 3.1257 - val_accuracy: 0.2614 - val_loss: 3.1334
Epoch 8/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.2672 - loss: 3.0560 - val_accuracy: 0.

#### **6.2 Performance Visualization**
**Instructions**: Plot the training and validation loss over time. Identify if the model is overfitting.


In [18]:
# TODO: Create a line chart for loss and accuracy
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

### **7. Testing the Chatbot**
#### **7.1 Inference Test**
**Instructions**: Write a prediction. Able to do string input, processes it, and returns the model's predicted response.


In [ ]:
# TODO: Write a code to do prediction answers for new questions
def generate_response(text):
    seq = tokenizer.texts_to_sequences([text])
    seq = pad_sequences(seq, maxlen=max_len, padding='post')

    pred = model.predict(seq, verbose=0)
    pred_indices = np.argmax(pred, axis=-1)[0]

    result = []
    for idx in pred_indices:
        if idx == 0:
            continue
        for word, index in tokenizer.word_index.items():
            if index == idx:
                result.append(word)
                break

    return " ".join(result)


# TODO: Test with 3 different questions about Cambodia Tourism and print the predicted answers
print("Question: where is angkor wat")
print("Answer:", generate_response("where is angkor wat"))
print()

print("Question: what food should i try in cambodia")
print("Answer:", generate_response("what food should i try in cambodia"))
print()

print("Question: best time to visit cambodia")
print("Answer:", generate_response("best time to visit cambodia"))

### **8. Conclusion & Reflection**
**Instructions**: Summarize your findings. What were the challenges with the SimpleRNN? How would you improve this chatbot?

### Challenges with SimpleRNN:

1. **Vanishing Gradient Problem**: SimpleRNN struggles to remember information
   from early parts of long sequences. When processing a 15-word question,
   information from the first few words may be "forgotten" by the time
   the model reaches the end.

2. **Limited Context Understanding**: The model predicts word-by-word based on
   patterns it memorized, not true language understanding. It cannot reason
   or combine concepts it hasn't seen together.

3. **Vocabulary Limitations**: If a user asks about something not in the
   training data (e.g., a specific hotel name), the model produces
   nonsensical output because those words have no learned representation.

4. **One-to-One Mapping**: The model essentially learns to map input patterns
   to output patterns. Similar questions may get completely different answers
   if the wording differs slightly.

### When Does the Chatbot Fail?

1. **Out-of-vocabulary words**: Questions with words not in training data
2. **Rephrased questions**: "What's the entry fee?" vs "How much is the ticket?"
3. **Complex questions**: Questions requiring reasoning or multiple facts
4. **Typos or misspellings**: "angkro wat" instead of "angkor wat"

### How Would I Improve This Chatbot?

1. Use LSTM or GRU instead of SimpleRNN (better at long sequences)
2. Use pre-trained embeddings (Word2Vec, GloVe)
3. Add attention mechanism
4. Use Transformer architecture (like BERT or GPT)
5. Expand training data with more variations of questions

In [ ]:
import pickle

# Save the trained model
model.save("model.h5")

# Save the tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Save config (max_len is critical for inference)
with open("config.pkl", "wb") as f:
    pickle.dump({"max_len": max_len}, f)

print("✅ Model, tokenizer, and config saved!")

In [ ]:
import os
print("Files saved in:", os.getcwd())
print("Files in folder:", os.listdir())